# Agent Comparison Analysis (RQ5)

This notebook analyzes how different AI agents (Claude Code, Cursor, Devin, Copilot, OpenAI Codex) differ in their library usage patterns.

In [ ]:
import sys

sys.path.append("..")

import json
import warnings
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

warnings.filterwarnings("ignore")

output_dir = Path("../output")
output_dir.mkdir(exist_ok=True)

## Load Data

In [ ]:
# Load results for each language
results_by_lang = {}

for lang in ["go", "python", "typescript"]:
    with open(output_dir / f"{lang}_library_usage.json", "r") as f:
        results_by_lang[lang.title()] = json.load(f)

print("Loaded results:")
for lang, results in results_by_lang.items():
    print(f"  {lang}: {len(results):,} PRs")

## Agent Distribution

In [ ]:
# Count PRs per agent per language
agent_counts = {}

for lang, results in results_by_lang.items():
    agent_counts[lang] = Counter([r["agent"] for r in results if r.get("agent")])

# Create DataFrame
df_agent_counts = pd.DataFrame(agent_counts).fillna(0).astype(int)
df_agent_counts["Total"] = df_agent_counts.sum(axis=1)
df_agent_counts = df_agent_counts.sort_values("Total", ascending=False)

print("\nAgent Distribution:")
print(df_agent_counts)

# Visualize
df_agent_counts.drop("Total", axis=1).plot(kind="barh", figsize=(10, 6))
plt.xlabel("Number of PRs")
plt.ylabel("Agent")
plt.title("Agent Distribution by Language")
plt.legend(title="Language")
plt.tight_layout()
plt.savefig(output_dir / "agent_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

## Per-Agent Statistics

Calculate key metrics for each agent across all languages.

In [ ]:
def calculate_agent_stats(results, agent_name):
    """Calculate statistics for a specific agent."""
    agent_prs = [r for r in results if r.get("agent") == agent_name]

    if not agent_prs:
        return None

    total = len(agent_prs)

    return {
        "total_prs": total,
        "prs_with_new_libs": sum(1 for r in agent_prs if r.get("new_libraries")),
        "pct_prs_with_new_libs": 100
        * sum(1 for r in agent_prs if r.get("new_libraries"))
        / total
        if total > 0
        else 0,
        "prs_with_dep_changes": sum(1 for r in agent_prs if r.get("dep_files_changed")),
        "pct_prs_with_dep_changes": 100
        * sum(1 for r in agent_prs if r.get("dep_files_changed"))
        / total
        if total > 0
        else 0,
        "total_unique_libs": len(
            set().union(*[set(r.get("all_libraries", [])) for r in agent_prs])
        ),
        "total_external_libs": len(
            set().union(*[set(r.get("external_libs", [])) for r in agent_prs])
        ),
        "total_stdlib_imports": len(
            set().union(*[set(r.get("stdlib_imports", [])) for r in agent_prs])
        ),
        "avg_libs_per_pr": sum(len(r.get("all_libraries", [])) for r in agent_prs)
        / total
        if total > 0
        else 0,
        "avg_new_libs_per_pr": sum(len(r.get("new_libraries", [])) for r in agent_prs)
        / total
        if total > 0
        else 0,
        "total_libs_with_version": sum(
            r.get("libs_with_version", 0) for r in agent_prs
        ),
        "total_libs_without_version": sum(
            r.get("libs_without_version", 0) for r in agent_prs
        ),
    }


# Calculate for each agent and language
agents = ["OpenAI_Codex", "Devin", "Copilot", "Cursor", "Claude_Code"]

agent_stats_by_lang = {}
for lang, results in results_by_lang.items():
    agent_stats_by_lang[lang] = {}
    for agent in agents:
        stats = calculate_agent_stats(results, agent)
        if stats:
            agent_stats_by_lang[lang][agent] = stats

# Also calculate combined stats across all languages
all_results = [r for results in results_by_lang.values() for r in results]
agent_stats_combined = {}
for agent in agents:
    stats = calculate_agent_stats(all_results, agent)
    if stats:
        agent_stats_combined[agent] = stats

print("Agent statistics calculated successfully!")

## Visualization: Agent Comparison Heatmap

In [ ]:
# Create heatmap comparing agents on key metrics
metrics = [
    ("pct_prs_with_new_libs", "% PRs with New Libs"),
    ("pct_prs_with_dep_changes", "% PRs with Dep Changes"),
    ("avg_libs_per_pr", "Avg Libs per PR"),
    ("avg_new_libs_per_pr", "Avg New Libs per PR"),
]

# Create matrix
heatmap_data = []
for agent in agents:
    if agent in agent_stats_combined:
        row = [agent_stats_combined[agent][metric[0]] for metric in metrics]
        heatmap_data.append(row)
    else:
        heatmap_data.append([0] * len(metrics))

df_heatmap = pd.DataFrame(heatmap_data, index=agents, columns=[m[1] for m in metrics])

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(df_heatmap, annot=True, fmt=".2f", cmap="YlOrRd", ax=ax)
plt.title("Agent Comparison: Library Usage Metrics")
plt.ylabel("Agent")
plt.tight_layout()
plt.savefig(output_dir / "agent_comparison_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

print("\nAgent Comparison (Combined):")
print(df_heatmap)

## Visualization: Grouped Bar Chart (Agents × Languages)

In [ ]:
# Create grouped bar chart: % PRs with new libraries by agent and language
data_for_plot = []

for lang in ["Go", "Python", "TypeScript"]:
    for agent in agents:
        if agent in agent_stats_by_lang.get(lang, {}):
            data_for_plot.append(
                {
                    "Language": lang,
                    "Agent": agent,
                    "Value": agent_stats_by_lang[lang][agent]["pct_prs_with_new_libs"],
                }
            )

df_plot = pd.DataFrame(data_for_plot)

# Plot
fig, ax = plt.subplots(figsize=(12, 6))
df_pivot = df_plot.pivot(index="Language", columns="Agent", values="Value")
df_pivot.plot(kind="bar", ax=ax, width=0.8)
plt.ylabel("% of PRs Adding New Libraries")
plt.xlabel("Language")
plt.title("Library Addition Rates by Agent and Language")
plt.legend(title="Agent", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(output_dir / "agent_language_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

## Agent-Specific Library Preferences

In [ ]:
# Find most common libraries for each agent
agent_library_prefs = {}

for agent in agents:
    agent_prs = [r for r in all_results if r.get("agent") == agent]

    if not agent_prs:
        continue

    # Count library frequency
    lib_counter = Counter()
    for pr in agent_prs:
        lib_counter.update(pr.get("all_libraries", []))

    agent_library_prefs[agent] = lib_counter.most_common(10)

# Display
for agent, libs in agent_library_prefs.items():
    print(f"\n{agent} - Top 10 Libraries:")
    for i, (lib, count) in enumerate(libs, 1):
        print(f"  {i:2d}. {lib:30s} {count:4d} PRs")

## Library Overlap Analysis

In [ ]:
# Calculate library overlap between agents
agent_libs = {}

for agent in agents:
    agent_prs = [r for r in all_results if r.get("agent") == agent]
    if agent_prs:
        agent_libs[agent] = set()
        for pr in agent_prs:
            agent_libs[agent].update(pr.get("all_libraries", []))

# Universal libraries (used by all agents)
if agent_libs:
    universal = set.intersection(*agent_libs.values())
    print(f"\nUniversal libraries (used by all agents): {len(universal)}")
    print(f"Top 10: {sorted(list(universal))[:10]}")

    # Agent-specific libraries (unique to each agent)
    print("\nAgent-specific libraries:")
    for agent, libs in agent_libs.items():
        other_agents = set.union(*[agent_libs[a] for a in agent_libs if a != agent])
        unique = libs - other_agents
        print(f"  {agent}: {len(unique)} unique libraries")
        if unique:
            print(f"    Examples: {list(unique)[:5]}")

## Statistical Summary Table

In [ ]:
# Create comprehensive summary table
summary_rows = []

for agent in agents:
    if agent in agent_stats_combined:
        s = agent_stats_combined[agent]
        total_versioned = s["total_libs_with_version"] + s["total_libs_without_version"]
        pct_versioned = (
            100 * s["total_libs_with_version"] / total_versioned
            if total_versioned > 0
            else 0
        )

        summary_rows.append(
            {
                "Agent": agent,
                "Total PRs": s["total_prs"],
                "% New Libs": f"{s['pct_prs_with_new_libs']:.1f}%",
                "% Dep Changes": f"{s['pct_prs_with_dep_changes']:.1f}%",
                "Avg Libs/PR": f"{s['avg_libs_per_pr']:.2f}",
                "Unique Libs": s["total_unique_libs"],
                "External": s["total_external_libs"],
                "Stdlib": s["total_stdlib_imports"],
                "% Versioned": f"{pct_versioned:.1f}%",
            }
        )

df_summary = pd.DataFrame(summary_rows)
print("\n" + "=" * 80)
print("AGENT COMPARISON SUMMARY")
print("=" * 80)
print(df_summary.to_string(index=False))

# Save to CSV
df_summary.to_csv(output_dir / "agent_comparison_summary.csv", index=False)
print(f"\nSaved to: {output_dir / 'agent_comparison_summary.csv'}")

## Export Results for Paper

In [ ]:
# Save detailed agent statistics for paper
output_data = {
    "agent_stats_combined": agent_stats_combined,
    "agent_stats_by_language": agent_stats_by_lang,
    "agent_library_preferences": {
        agent: [(lib, count) for lib, count in libs]
        for agent, libs in agent_library_prefs.items()
    },
    "agent_distribution": df_agent_counts.to_dict(),
}

with open(output_dir / "agent_comparison_analysis.json", "w") as f:
    json.dump(output_data, f, indent=2)

print("\nAgent comparison analysis complete!")
print(f"Results saved to: {output_dir / 'agent_comparison_analysis.json'}")

## Key Insights for Paper

Based on the analysis above, we can answer **RQ5: How does library usage vary across different AI coding agents?**

### Expected Findings:

1. **Variation in conservatism**: Different agents show different rates of adding new libraries
2. **Language interaction**: Agent behavior may depend on the programming language
3. **Version specification practices**: Some agents may be more diligent about versions
4. **Library preferences**: Agents may favor different libraries based on training data
5. **Universal vs unique libraries**: Some libraries are used by all agents, others are agent-specific

### For the Paper:
- Include the heatmap as **Figure X**
- Include the grouped bar chart as **Figure Y**
- Include the summary table as **Table Z**
- Dedicate 0.3 pages to RQ5 results
- Discuss implications for agent developers and users